# 06 — Scientific evaluation

**Purpose:** Validate prediction coverage, source hashes and label set before reporting macro-F1, accuracy and invalid-output rates.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# GitHub setup: run this as the FIRST code cell in Kaggle or Colab.
# This clones the source code only; GPU checkpoints/results need separate storage.
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
REPO_FOLDER = "nlp_research_proj"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # True: update an existing clean clone from GitHub

if Path("/kaggle/working").is_dir():
    workspace = Path("/kaggle/working")
elif Path("/content").is_dir():
    workspace = Path("/content")
else:
    workspace = Path.cwd()

repo_dir = workspace / REPO_FOLDER
project_dir = repo_dir / PROJECT_FOLDER

# Local development support: use the existing source folder outside Kaggle/Colab.
local_project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "project_utils.py").is_file()), None)
if local_project is not None and not (Path("/kaggle/working").is_dir() or Path("/content").is_dir()):
    ROOT = local_project
else:
    if not repo_dir.exists():
        print("Cloning:", REPO_URL)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)])
    elif PULL_LATEST:
        # Only fast-forward; protects existing local changes from being overwritten.
        subprocess.check_call(["git", "-C", str(repo_dir), "pull", "--ff-only"])
    else:
        print("Using existing clone:", repo_dir)

    ROOT = project_dir

if not (ROOT / "project_utils.py").is_file():
    raise FileNotFoundError(
        f"Cannot find project_utils.py at {ROOT}. Confirm the GitHub folder name."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
print("Project loaded from:", ROOT)


This notebook is deliberately strict. It rejects predictions for the wrong task, partial samples, wrong dataset versions, unknown example IDs and duplicate IDs. Invalid generated labels are counted as mistakes, not silently discarded.

In [ ]:
import pandas as pd
TASK='sib200'; LANG='ur'; SPLIT='validation'  # switch to 'test' only at final evaluation
paths=sorted((ROOT/'outputs/predictions').glob(f'*_{LANG}_{SPLIT}.csv'))
print('Found',len(paths),'prediction files')
for p in paths: print(p.name)

In [ ]:
results=[]
rows=load_task(TASK,LANG,SPLIT); labels=task_labels(TASK)
for p in paths:
    meta_path=p.with_suffix('.meta.json')
    if not meta_path.exists(): print('SKIP (no metadata):',p); continue
    meta=read_json(meta_path)
    if meta['task']!=TASK: print('SKIP other task:',p.name); continue
    try:
        validate_prediction_metadata(meta,TASK,LANG,SPLIT)
        if meta.get('subset'): raise ValueError('partial/smoke prediction set cannot be called full evaluation')
        predictions=read_csv(p)
        score=evaluate_predictions(rows,predictions,labels)
        compact={k:v for k,v in score.items() if k not in ['classification_report','confusion_matrix','confusion_labels']}
        compact.update({'run_id':meta['run_id'],'method':meta.get('method',meta['run_id']),
                        'task':TASK,'lang':LANG,'split':SPLIT,'file':str(p)})
        results.append(compact)
        write_json(ROOT/'outputs/metrics'/f'{p.stem}_metrics.json',dict(**compact, details=score, metadata=meta))
        print(p.name,'accuracy',round(score['accuracy'],4),'macro-F1',round(score['macro_f1'],4),
              'invalid',score['invalid_count'])
    except (ValueError,KeyError,FileNotFoundError) as e:
        print('REJECTED',p.name,'reason:',e)
if results:
    df=pd.DataFrame(results).sort_values('macro_f1',ascending=False)
    display(df[['run_id','split','count','accuracy','macro_f1','invalid_count']])
    df.to_csv(ROOT/'outputs/metrics'/f'{TASK}_{LANG}_{SPLIT}_leaderboard.csv',index=False)
else: print('No complete, verified predictions available yet.')